<a href="https://colab.research.google.com/github/diegocortes30/olist_ecommerce_analysis/blob/main/03_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 – Data Cleaning

**Project:** Olist Brazilian E-Commerce Analysis

**Goal:** clean each of the Olist tables so they are reliable for the analysis and the dashboard.

**Input:** the 9 original CSV files in `raw/` (never modified).

**Output:** 8 clean CSV files in `processed/`.

**How this notebook works:** each table follows the same cycle: look at the problem → decide what to do and why → apply it → check the result. Nothing is deleted without a reason. When a value is wrong or missing, it is either marked with a flag column, kept as null, or labeled (for example `unknown`), so no data is invented.

**Not included:** the tables are cleaned one by one here and are **not joined**. The join (one row per order) is done in the analysis notebook.

## 1. Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd

RAW = '/content/drive/MyDrive/Olist/raw'
PROCESSED = '/content/drive/MyDrive/Olist/processed'
os.makedirs(PROCESSED, exist_ok=True)

Mounted at /content/drive


In [3]:
orders = pd.read_csv('/content/drive/MyDrive/Olist/raw/olist_orders_dataset.csv')
customers = pd.read_csv('/content/drive/MyDrive/Olist/raw/olist_customers_dataset.csv')
geolocation = pd.read_csv('/content/drive/MyDrive/Olist/raw/olist_geolocation_dataset.csv')
order_items = pd.read_csv('/content/drive/MyDrive/Olist/raw/olist_order_items_dataset.csv')
order_payments = pd.read_csv('/content/drive/MyDrive/Olist/raw/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('/content/drive/MyDrive/Olist/raw/olist_order_reviews_dataset.csv')
products = pd.read_csv('/content/drive/MyDrive/Olist/raw/olist_products_dataset.csv')
sellers = pd.read_csv('/content/drive/MyDrive/Olist/raw/olist_sellers_dataset.csv')
product_category_name_translation = pd.read_csv('/content/drive/MyDrive/Olist/raw/product_category_name_translation.csv')

## 2. Dataframes cleaning

### 2.1. orders df

In [4]:
# Look at the dataframe shape
orders.shape

(99441, 8)

In [5]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [6]:
# Look at the dtypes of the columns and look for changes
orders.dtypes

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,object
order_approved_at,object
order_delivered_carrier_date,object
order_delivered_customer_date,object
order_estimated_delivery_date,object


In [7]:
# Change the type of the date columns
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])
orders.dtypes

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]


In [8]:
# Check if there are null values
orders.isnull().sum()

,0
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


In [9]:
# Create a df with only the delivered orders to check the nulls
delivered = orders[orders['order_status'] == 'delivered']
delivered.isnull().sum()

,0
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,14
order_delivered_carrier_date,2
order_delivered_customer_date,8
order_estimated_delivery_date,0


In [10]:
# Create a new column in 'orders' that says True when the status is 'delivered' and a date is missing
orders['missing_delivery_date'] = (orders['order_status'] == 'delivered') & (orders[['order_delivered_customer_date', 'order_approved_at', 'order_delivered_carrier_date']].isnull().any(axis=1))
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,missing_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,False


In [11]:
orders['missing_delivery_date'].sum()

np.int64(23)

In [12]:
# Look for duplicates
orders.duplicated().sum()

np.int64(0)

In [13]:
# Duplicates in order_id
orders['order_id'].duplicated().sum()

np.int64(0)

In [14]:
# Look if the dates makes sense
(orders['order_delivered_customer_date'] < orders['order_purchase_timestamp']).sum()

np.int64(0)

In [15]:
# Check 'order_status' values
orders['order_status'].value_counts()

,count
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [16]:
# Save the new dataframe
orders.to_csv(f'{PROCESSED}/orders_clean.csv', index=False)

Findings:


*   5 'object' columns were converted to 'datetime'
*   Most of the nulls are because of the order status. The 23 delivered orders with a missing value where kept as nulls and are marked in the column 'missing_delivery_date'
*   There are no duplicates in the dataframe 'orders' and in the column 'order_id'
*   There are no delivered dates that make no sense
*   The 8 status are valid

### 2.2. order_items dataframe

In [17]:
# Check the df shape
order_items.shape

(112650, 7)

In [18]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [19]:
# Check the dtypes
order_items.dtypes

,0
order_id,object
order_item_id,int64
product_id,object
seller_id,object
shipping_limit_date,object
price,float64
freight_value,float64


In [20]:
# Change object type to date
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])
order_items.dtypes

,0
order_id,object
order_item_id,int64
product_id,object
seller_id,object
shipping_limit_date,datetime64[ns]
price,float64
freight_value,float64


In [21]:
# Look for null values
order_items.isnull().sum()

,0
order_id,0
order_item_id,0
product_id,0
seller_id,0
shipping_limit_date,0
price,0
freight_value,0


In [22]:
# Look for duplicates
order_items.duplicated().sum()

np.int64(0)

In [23]:
# Duplicates in 'order_id' and 'order_item_id'
order_items[['order_id', 'order_item_id']].duplicated().sum()

np.int64(0)

In [24]:
# Check the price columns
round(order_items[['price', 'freight_value']].describe(),2)

,price,freight_value
count,112650.00,112650.00
mean,120.65,19.99
std,183.63,15.81
min,0.85,0.00
25%,39.90,13.08
50%,74.99,16.26
75%,134.90,21.15
max,6735.00,409.68


In [25]:
# Check te max values
order_items['price'].sort_values(ascending=False).head()

,price
3556,6735.0
112233,6729.0
107841,6499.0
74336,4799.0
11249,4690.0


In [26]:
# Check how many freight values = 0
(order_items['freight_value'] == 0).sum()

np.int64(383)

In [27]:
# Save the new dataframe
order_items.to_csv(f'{PROCESSED}/order_items_clean.csv', index=False)

Findings:
*   'shipping_limit_date' was as an object type, it was changed to 'datetime'
*   There are no null values
*   There are no duplicates in the dataframe
*   'price' column have no 0 or negative values, the high values (up to 6,735) make sense as expensive products
*   383 items have freight_value as 0, probably because the free shipping, no changes were made

### 2.3. customers dataframe

In [28]:
# Check the shape
customers.shape

(99441, 5)

In [29]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [30]:
# Check the dtypes
customers.dtypes

,0
customer_id,object
customer_unique_id,object
customer_zip_code_prefix,int64
customer_city,object
customer_state,object


In [31]:
# Change zip code to string
customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].astype(str)
customers.dtypes

,0
customer_id,object
customer_unique_id,object
customer_zip_code_prefix,object
customer_city,object
customer_state,object


In [32]:
# Fix the zip code adding a zero before the first number
customers['customer_zip_code_prefix']= customers['customer_zip_code_prefix'].str.zfill(5)
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [33]:
# Check if all the zip codes have 5 digits
customers['customer_zip_code_prefix'].str.len().value_counts()

,count
customer_zip_code_prefix,
5,99441


In [34]:
# Look for null values
customers.isnull().sum()

,0
customer_id,0
customer_unique_id,0
customer_zip_code_prefix,0
customer_city,0
customer_state,0


In [35]:
# Look for duplicates
customers.duplicated().sum()

np.int64(0)

In [36]:
customers['customer_id'].duplicated().sum()

np.int64(0)

In [37]:
# Look for unique values in customer_unique_id
customers['customer_unique_id'].nunique()

96096

In [38]:
(customers['customer_unique_id'].value_counts() > 1).sum()

np.int64(2997)

In [39]:
# Look for clients in each state
customers['customer_state'].value_counts()

,count
customer_state,
SP,41746
RJ,12852
MG,11635
RS,5466
PR,5045
SC,3637
BA,3380
DF,2140
ES,2033


In [40]:
# Look for unique values in cities
customers['customer_city'].nunique()

4119

In [41]:
customers['customer_city'].str.lower().str.strip().nunique()

4119

In [42]:
# Save the new dataframe
customers.to_csv(f'{PROCESSED}/customers_clean.csv', index=False)

Findings:

*   The zip code prefixes were as numbers type and that cause the initial zero to get lost. The column were converted to text to fill the 5 digits
*   There are no nulls or duplicates
*   'customer_id' is unique for order. There are 96,096 unique clients and 2,997 of them (3.1%) bought more than once
*   'customer_state' have 27 valid values
*   'customer_city' have no cap variants or space variants

### 2.4. order_payments dataframe

In [43]:
# Check the shape
order_payments.shape

(103886, 5)

In [44]:
order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [45]:
# Check dtypes
order_payments.dtypes

,0
order_id,object
payment_sequential,int64
payment_type,object
payment_installments,int64
payment_value,float64


In [46]:
# Look for nulls
order_payments.isnull().sum()

,0
order_id,0
payment_sequential,0
payment_type,0
payment_installments,0
payment_value,0


In [47]:
# Look for duplictates
order_payments.duplicated().sum()

np.int64(0)

In [48]:
# Look for duplicates in order_id and payment_sequential
order_payments[['order_id', 'payment_sequential']].duplicated().sum()

np.int64(0)

In [49]:
# Check value counts in payment_types
order_payments['payment_type'].value_counts()

,count
payment_type,
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529
not_defined,3


In [50]:
# Look for the not_defined rows
order_payments[order_payments['payment_type'] == 'not_defined']

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [51]:
# Look for all the values with payment_value = 0
order_payments[order_payments['payment_value'] == 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [52]:
# Look for all the values with payment_installments = 0
order_payments[order_payments['payment_installments'] == 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [53]:
# Add these new colums to the dataframe
order_payments['zero_value_payment'] = (order_payments['payment_value'] == 0)
order_payments['zero_installments_payment'] = (order_payments['payment_installments'] == 0)

In [54]:
# Check the descriptive values
round(order_payments[['payment_value', 'payment_installments', 'payment_sequential']].describe(),2)

,payment_value,payment_installments,payment_sequential
count,103886.00,103886.00,103886.00
mean,154.10,2.85,1.09
std,217.49,2.69,0.71
min,0.00,0.00,1.00
25%,56.79,1.00,1.00
50%,100.00,1.00,1.00
75%,171.84,4.00,1.00
max,13664.08,24.00,29.00


In [55]:
# Check te max values for payment_value
order_payments['payment_value'].sort_values(ascending=False).head()

,payment_value
52107,13664.08
34370,7274.88
41419,6929.31
49581,6922.21
85539,6726.66


In [56]:
# Save the new dataframe
order_payments.to_csv(f'{PROCESSED}/order_payments_clean.csv', index=False)

Findings:
*   103,886 rows with correct dtypes and no null values
*   There are no duplicates in the dataframe
*   'payment_type' have 5 values. 'not_defined' appears only 3 times, all with 0 value
*   9 payments have 0 value (6 vouchers and 3 not defined) and 2 card payments have 0 installments with real values. Nothing was elimiated: 2 new columns were created 'zero_value_payment' and 'zero_installments_payment'
*   There are no negative values, the maximum value (13,664.08) is reasonable.

### 2.5. order_reviews

In [57]:
# Check the shape
order_reviews.shape

(99224, 7)

In [58]:
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [59]:
# Check the dtypes
order_reviews.dtypes

,0
review_id,object
order_id,object
review_score,int64
review_comment_title,object
review_comment_message,object
review_creation_date,object
review_answer_timestamp,object


In [60]:
# Change object for date
order_reviews['review_creation_date'] = pd.to_datetime(order_reviews['review_creation_date'])
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'])
order_reviews.dtypes

,0
review_id,object
order_id,object
review_score,int64
review_comment_title,object
review_comment_message,object
review_creation_date,datetime64[ns]
review_answer_timestamp,datetime64[ns]


In [61]:
# Look for null values
order_reviews.isnull().sum()

,0
review_id,0
order_id,0
review_score,0
review_comment_title,87656
review_comment_message,58247
review_creation_date,0
review_answer_timestamp,0


In [62]:
# Look for duplicates
order_reviews.duplicated().sum()

np.int64(0)

In [63]:
order_reviews['review_id'].duplicated().sum()

np.int64(814)

In [64]:
order_reviews[['order_id', 'review_id']].duplicated().sum()

np.int64(0)

In [65]:
order_reviews['order_id'].duplicated().sum()

np.int64(551)

In [66]:
# Look for unique values
order_reviews['order_id'].nunique()

98673

In [67]:
# Check value counts in review_score
order_reviews['review_score'].value_counts()

,count
review_score,
5,57328
4,19142
1,11424
3,8179
2,3151


In [68]:
# Check if the dates make sense
(order_reviews['review_answer_timestamp'] < order_reviews['review_creation_date']).sum()

np.int64(0)

In [69]:
# Save the new dataframe
order_reviews.to_csv(f'{PROCESSED}/order_reviews_clean.csv', index=False)

Findings:
*  The date columns with object dtype were changed as datetime.
*  Null values are only find in comments: 87,656 on the title and 58,247 on the message. These are not errors, there are reviews without text and we kept them as nulls
*   There are no duplicated rows. 'review_id' repeats in 814 rows,but the pair 'review_id' + 'order_id' is unique, so we don't delete anything
*   551 rows are orders with more than one review (98,673 distinct orders)
*   'review_score' only have values between 1 to 5
*   There are no reviews before the creation date

### 2.6. products dataframe

In [70]:
# Check  the shape
products.shape

(32951, 9)

In [71]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [72]:
# Check the dtypes
products.dtypes

,0
product_id,object
product_category_name,object
product_name_lenght,float64
product_description_lenght,float64
product_photos_qty,float64
product_weight_g,float64
product_length_cm,float64
product_height_cm,float64
product_width_cm,float64


In [73]:
# Look for nulls
products.isnull().sum()

,0
product_id,0
product_category_name,610
product_name_lenght,610
product_description_lenght,610
product_photos_qty,610
product_weight_g,2
product_length_cm,2
product_height_cm,2
product_width_cm,2


In [74]:
# Filter null products
no_category = products[products['product_category_name'].isnull()]
no_category

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


In [75]:
# Count the nulls
no_category.isnull().sum()

,0
product_id,0
product_category_name,610
product_name_lenght,610
product_description_lenght,610
product_photos_qty,610
product_weight_g,1
product_length_cm,1
product_height_cm,1
product_width_cm,1


In [76]:
# Fill thne nulls in category_name
products['product_category_name'] = products['product_category_name'].fillna('unknown')
products['product_category_name'].isnull().sum()

np.int64(0)

In [77]:
(products['product_category_name'] == 'unknown').sum()

np.int64(610)

In [78]:
# Look for duplicates
products.duplicated().sum()

np.int64(0)

In [79]:
products['product_id'].duplicated().sum()

np.int64(0)

In [80]:
# Check the descriptive values
round(products[['product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']].describe(),2)

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.00,32341.00,32341.00,32949.00,32949.00,32949.00,32949.00
mean,48.48,771.50,2.19,2276.47,30.82,16.94,23.20
std,10.25,635.12,1.74,4282.04,16.91,13.64,12.08
min,5.00,4.00,1.00,0.00,7.00,2.00,6.00
25%,42.00,339.00,1.00,300.00,18.00,8.00,15.00
50%,51.00,595.00,1.00,700.00,25.00,13.00,20.00
75%,57.00,972.00,3.00,1900.00,38.00,21.00,30.00
max,76.00,3992.00,20.00,40425.00,105.00,105.00,118.00


In [81]:
# Count product_weight_g with value 0
(products['product_weight_g'] == 0).sum()

np.int64(4)

In [82]:
# Replace the values that weight 0 with nan
import numpy as np

products['product_weight_g'] = products['product_weight_g'].replace(0, np.nan)
(products['product_weight_g'] == 0).sum()

np.int64(0)

In [83]:
products['product_weight_g'].isnull().sum()

np.int64(6)

In [84]:
# Save the new dataframe
products.to_csv(f'{PROCESSED}/products_clean.csv', index=False)

Findings:
*   32,951 products have no duplicates, including 'product_id'
*   There are no catalogue information in 610 products (category, length name, description lenght and photos). The category was filled with 'unknown' and the other columns were kept as nulls
*   4 products have 0 weight, that weight was converted to null. There are 6 weight nulls in total
*   The rest of the extreme values (up to 40 kg and 105 x 105 x 118 cm) are reasonable


### 2.7. category_translation dataframe

In [85]:
# Check the shape
product_category_name_translation.shape

(71, 2)

In [86]:
product_category_name_translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [87]:
# Check the dtypes
product_category_name_translation.dtypes

,0
product_category_name,object
product_category_name_english,object


In [88]:
# Look for nulls
product_category_name_translation.isnull().sum()

,0
product_category_name,0
product_category_name_english,0


In [89]:
# Look for duplicates
product_category_name_translation.duplicated().sum()

np.int64(0)

In [90]:
product_category_name_translation['product_category_name'].duplicated().sum()

np.int64(0)

In [91]:
# Find categories without transalation with isin
products[~products['product_category_name'].isin(product_category_name_translation['product_category_name'])]['product_category_name'].value_counts()

,count
product_category_name,
unknown,610
portateis_cozinha_e_preparadores_de_alimentos,10
pc_gamer,3


In [92]:
# Create a table with the categories missing
new_rows = pd.DataFrame({
    'product_category_name': ['portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer', 'unknown'],
    'product_category_name_english': ['portable_kitchen_and_food_preparers', 'pc_gamer', 'unknown']
})

product_category_name_translation = pd.concat(
    [product_category_name_translation, new_rows], ignore_index=True)
product_category_name_translation.shape

(74, 2)

In [93]:
product_category_name_translation['product_category_name'].duplicated().sum()

np.int64(0)

In [94]:
products[~products['product_category_name'].isin(product_category_name_translation['product_category_name'])]['product_category_name'].value_counts()

,count
product_category_name,


In [95]:
# Save the new dataframe
product_category_name_translation.to_csv(f'{PROCESSED}/product_category_name_translation_clean.csv', index=False)

Findings:
*   71 categories have no nulls or duplicated values
*   2 product categories were missing ('portable_kitchen_and_food_preparers' and 'pc_gamer', 13 products) and the unknown label (610 products). The categories were added and the table ended up with 74 rows.


### 2.8. sellers dataframe

In [96]:
# Check the shape
sellers.shape

(3095, 4)

In [97]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [98]:
# Check the dtypes
sellers.dtypes

,0
seller_id,object
seller_zip_code_prefix,int64
seller_city,object
seller_state,object


In [99]:
# Add a zero in the zip code so every zip code can have 5 digits
sellers['seller_zip_code_prefix'] = sellers['seller_zip_code_prefix'].astype(str).str.zfill(5)
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [100]:
# Look for null values
sellers.isnull().sum()

,0
seller_id,0
seller_zip_code_prefix,0
seller_city,0
seller_state,0


In [101]:
# Look for duplicated values
sellers.duplicated().sum()

np.int64(0)

In [102]:
# Look for duplicated values in seller_id
sellers['seller_id'].duplicated().sum()

np.int64(0)

In [103]:
# Confirm every zip code have 5 digits
sellers['seller_zip_code_prefix'].str.len().value_counts()

,count
seller_zip_code_prefix,
5,3095


In [104]:
# Count sellers per state
sellers['seller_state'].value_counts()

,count
seller_state,
SP,1849
PR,349
MG,244
SC,190
RJ,171
RS,129
GO,40
DF,30
ES,23


In [105]:
# Count unique values in seller city
sellers['seller_city'].nunique()

611

In [106]:
sellers['seller_city'].str.lower().str.strip().nunique()

611

In [107]:
# Look for names with odd characters
sellers[sellers['seller_city'].str.contains(r'[^a-z\s]')]

,seller_id,seller_zip_code_prefix,seller_city,seller_state
78,731ef20c231d9a7103a425e83fd91271,88501,lages - sc,SC
237,c3aad7dc65449ae90a5e9c3c6c1e78e0,15350,auriflama/sp,SP
246,71593c7413973a1e160057b80d4958f6,03407,sao paulo / sao paulo,SP
360,a3fa18b3f688ec0fca3eb8bfcbd2d5b3,04557,são paulo,SP
476,26b482dccfa29bd2e40703ba45523702,13450,santa barbara d´oeste,SP
517,ceb7b4fb9401cd378de7886317ad1b47,22790,04482255,RJ
551,723a46b89fd5c3ed78ccdf039e33ac63,93310,"novo hamburgo, rio grande do sul, brasil",RS
622,7994b065a7ffb14e71c6312cf87b9de2,29142,cariacica / es,ES
707,11938fcc75f6902fea1d0a6f72b54a57,89900,sao miguel d'oeste,SC
826,05e107217c7266362fd44b75b2cd4cc4,05353,sao paulo - sp,SP


In [108]:
# Make a dictionary with the corrections
city_corrections = {
    'lages - sc': 'lages',
    'auriflama/sp': 'auriflama',
    'sao paulo / sao paulo': 'sao paulo',
    'são paulo': 'sao paulo',
    'santa barbara d´oeste': "santa barbara d'oeste",
    '04482255': np.nan,
    'novo hamburgo, rio grande do sul, brasil': 'novo hamburgo',
    'cariacica / es': 'cariacica',
    'sao paulo - sp': 'sao paulo',
    'sbc/sp': 'sao bernardo do campo',
    'santo andre/sao paulo': 'santo andre',
    'sp / sp': 'sao paulo',
    'maua/sao paulo': 'maua',
    'mogi das cruzes / sp': 'mogi das cruzes',
    'rio de janeiro \\rio de janeiro': 'rio de janeiro',
    'barbacena/ minas gerais': 'barbacena',
    'andira-pr': 'andira',
    'rio de janeiro / rio de janeiro': 'rio de janeiro',
    'pinhais/pr': 'pinhais',
    'ribeirao preto / sao paulo': 'ribeirao preto',
    'carapicuiba / sao paulo': 'carapicuiba',
    'vendas@creditparts.com.br': np.nan,
    'sao sebastiao da grama/sp': 'sao sebastiao da grama',
    'rio de janeiro, rio de janeiro, brasil': 'rio de janeiro',
    'jacarei / sao paulo': 'jacarei',
}

sellers['seller_city'] = sellers['seller_city'].replace(city_corrections)

In [109]:
# Confirm the changes
sellers[sellers['seller_city'].str.contains(r'[^a-z\s]', na=False)]

,seller_id,seller_zip_code_prefix,seller_city,seller_state
360,a3fa18b3f688ec0fca3eb8bfcbd2d5b3,04557,são paulo,SP
476,26b482dccfa29bd2e40703ba45523702,13450,santa barbara d'oeste,SP
707,11938fcc75f6902fea1d0a6f72b54a57,89900,sao miguel d'oeste,SC
874,4aba391bc3b88717ce08eb11e44937b2,45816,arraial d'ajuda (porto seguro),BA
1890,30a2f535bb48308f991d0b9ad4a8c4bb,13457,santa barbara d'oeste,SP
1919,13dd308f81fea30cc670c656b2b46cc3,13454,santa barbara d'oeste,SP
1982,20f0aeea30bc3b8c4420be8ced4226c0,13456,santa barbara d'oeste,SP
2036,54a1852d1b8f10312c55e906355666ee,13456,santa barbara d'oeste,SP
2340,5dceca129747e92ff8ef7a997dc4f8ca,13450,santa barbara d'oeste,SP


In [110]:
sellers['seller_city'] = (sellers['seller_city']
                          .str.normalize('NFKD')
                          .str.encode('ascii', errors='ignore')
                          .str.decode('utf-8'))

In [111]:
sellers[sellers['seller_city'].str.contains(r'[^a-z\s]', na=False)]

,seller_id,seller_zip_code_prefix,seller_city,seller_state
476,26b482dccfa29bd2e40703ba45523702,13450,santa barbara d'oeste,SP
707,11938fcc75f6902fea1d0a6f72b54a57,89900,sao miguel d'oeste,SC
874,4aba391bc3b88717ce08eb11e44937b2,45816,arraial d'ajuda (porto seguro),BA
1890,30a2f535bb48308f991d0b9ad4a8c4bb,13457,santa barbara d'oeste,SP
1919,13dd308f81fea30cc670c656b2b46cc3,13454,santa barbara d'oeste,SP
1982,20f0aeea30bc3b8c4420be8ced4226c0,13456,santa barbara d'oeste,SP
2036,54a1852d1b8f10312c55e906355666ee,13456,santa barbara d'oeste,SP
2340,5dceca129747e92ff8ef7a997dc4f8ca,13450,santa barbara d'oeste,SP


In [112]:
sellers['seller_city'].isnull().sum()

np.int64(2)

In [113]:
(sellers['seller_city'] == 'sao paulo').sum()

np.int64(700)

In [114]:
# Save the new dataframe
sellers.to_csv(f'{PROCESSED}/sellers_clean.csv', index=False)

Findings:
*   3,095 sellers, without null or duplicate values ('seller_id' is unique)
*   The zip prefixs were converted to string and fill with 5 digits so the 0 can be added.
*   'seller_states' have valid codes of 23 states. SP concentrates almost the 60% of the sellers
*   'seller_city' had 34 entries with odd format (abreviations, a number, a mail, accents). A dictionary with corrections was made to normalize the entries.

## 3. Summary of the cleaning

| Table | Rows | Main issues found | Decisions | Output file |
|---|---|---|---|---|
| `orders` | 99,441 | Dates as text; nulls in date columns; 23 delivered orders with a missing date | Dates converted to `datetime`; nulls kept; flag column `missing_delivery_date` | `orders_clean.csv` |
| `order_items` | 112,650 | `shipping_limit_date` as text; 383 items with `freight_value` = 0 | Date converted; zero freight kept (likely free shipping) | `order_items_clean.csv` |
| `customers` | 99,441 | Zip codes as numbers (leading zero lost) | Converted to text and padded to 5 digits | `customers_clean.csv` |
| `order_payments` | 103,886 | 9 payments with value 0; 2 card payments with 0 installments | Nothing deleted; flag columns `zero_value_payment` and `zero_installments_payment` | `order_payments_clean.csv` |
| `order_reviews` | 99,224 | Dates as text; `review_id` repeated; 551 rows from orders with more than one review | Dates converted; comment nulls kept; duplicates handled at join time | `order_reviews_clean.csv` |
| `products` | 32,951 | 610 products without category; 4 products with weight 0 | Category filled with `unknown`; weight 0 converted to null | `products_clean.csv` |
| `product_category_name_translation` | 74 | 2 categories without translation (13 products) plus `unknown` | 3 rows added manually | `product_category_name_translation_clean.csv` |
| `sellers` | 3,095 | Zip codes as numbers; 34 odd city names | Zip padded to 5 digits; city dictionary of corrections; 2 invalid entries set to null | `sellers_clean.csv` |
| `geolocation` | – | Not cleaned (see note below) | Left out on purpose | – |

### **Note:**
**geolocation dataframe — not cleaned:**

The five business questions don’t need coordinates. It can be cleaned later if the analysis needs maps or seller–customer distances.